In [1]:
import pandas as pd
import os

DATA_DIR = "."
dfs = {}  # key = location id, value = DataFrame

# Load all CSVs first
for loc in range(1, 24):
    filename = f"exportloc{loc}IQ.csv"
    filepath = os.path.join(DATA_DIR, filename)

    if os.path.exists(filepath):
        dfs[loc] = pd.read_csv(filepath)
    else:
        print(f"Missing file: {filename}")

# Initialize common columns from first available location
first_loc = next(iter(dfs))
common_columns = set(dfs[first_loc].columns)

# Find intersection
for loc in dfs:
    common_columns &= set(dfs[loc].columns)

common_columns = sorted(common_columns)

print("Number of common features:", len(common_columns))
print(common_columns)

Number of common features: 1057
['CIR0', 'CIR1', 'CIR10', 'CIR100', 'CIR1000', 'CIR1001', 'CIR1002', 'CIR1003', 'CIR1004', 'CIR1005', 'CIR1006', 'CIR1007', 'CIR1008', 'CIR1009', 'CIR101', 'CIR1010', 'CIR1011', 'CIR1012', 'CIR1013', 'CIR1014', 'CIR1015', 'CIR102', 'CIR103', 'CIR104', 'CIR105', 'CIR106', 'CIR107', 'CIR108', 'CIR109', 'CIR11', 'CIR110', 'CIR111', 'CIR112', 'CIR113', 'CIR114', 'CIR115', 'CIR116', 'CIR117', 'CIR118', 'CIR119', 'CIR12', 'CIR120', 'CIR121', 'CIR122', 'CIR123', 'CIR124', 'CIR125', 'CIR126', 'CIR127', 'CIR128', 'CIR129', 'CIR13', 'CIR130', 'CIR131', 'CIR132', 'CIR133', 'CIR134', 'CIR135', 'CIR136', 'CIR137', 'CIR138', 'CIR139', 'CIR14', 'CIR140', 'CIR141', 'CIR142', 'CIR143', 'CIR144', 'CIR145', 'CIR146', 'CIR147', 'CIR148', 'CIR149', 'CIR15', 'CIR150', 'CIR151', 'CIR152', 'CIR153', 'CIR154', 'CIR155', 'CIR156', 'CIR157', 'CIR158', 'CIR159', 'CIR16', 'CIR160', 'CIR161', 'CIR162', 'CIR163', 'CIR164', 'CIR165', 'CIR166', 'CIR167', 'CIR168', 'CIR169', 'CIR17', 'CI

In [2]:
import pandas as pd

dtype_map = {}  # column -> {dtype: [locations]}

for loc, df in dfs.items():
    for col in common_columns:
        dtype = df[col].dtype
        dtype_map.setdefault(col, {}).setdefault(dtype, []).append(loc)

# Find columns with inconsistent dtypes
inconsistent = {
    col: types
    for col, types in dtype_map.items()
    if len(types) > 1
}

print("Number of columns with inconsistent dtypes:", len(inconsistent))

# Show first few problematic columns
for col, types in list(inconsistent.items())[:10]:
    print(f"\nColumn: {col}")
    for dtype, locs in types.items():
        print(f"  {dtype}: locations {locs}")

Number of columns with inconsistent dtypes: 0


In [6]:
"""
=============================================================================
STEP 2: DATA AUDIT & PREPARATION
Run this after confirming 0 inconsistent dtypes.
This will tell you exactly what your data looks like before modelling.
=============================================================================
"""

# ── Assumes dfs and common_columns already exist from your earlier cell ──────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import re

# ── 1. BASIC AUDIT ────────────────────────────────────────────────────────────
print("="*65)
print("  DATA AUDIT")
print("="*65)

total_rows = 0
for loc, df in sorted(dfs.items()):
    n     = len(df)
    total_rows += n

    # Error stats
    err   = pd.to_numeric(df["error"].replace("-", np.nan), errors="coerce")
    n_nan = err.isna().sum()
    err   = err.fillna(0)

    # LOS ratio
    los_col = pd.to_numeric(df["LOS"].replace("-", np.nan), errors="coerce")
    los_pct = (los_col == 1).mean() * 100 if "LOS" in df.columns else float("nan")

    # CIR dtype check for this location
    cir_cols  = [c for c in df.columns if c.startswith("CIR") and c[3:].isdigit()]
    cir_dtype = df[cir_cols[0]].dtype if cir_cols else "N/A"

    print(f"  Loc {loc:2d}: {n:6d} rows | "
          f"err mean={err.mean():+7.1f}  std={err.std():6.1f}  "
          f"|err| mean={err.abs().mean():6.1f} mm | "
          f"NaN err: {n_nan:3d} | LOS%: {los_pct:5.1f} | "
          f"CIR dtype: {cir_dtype}")

print(f"\n  Total rows across all locations: {total_rows:,}")
print(f"  Total CIR columns found: {len(cir_cols)}")
print(f"  Common columns: {len(common_columns)}")

# ── 2. SAMPLE ONE CIR VALUE TO CONFIRM FORMAT ────────────────────────────────
print("\n" + "="*65)
print("  CIR VALUE FORMAT CHECK (first non-null value from each dtype category)")
print("="*65)

sample_loc = list(dfs.keys())[0]
sample_df  = dfs[sample_loc]
cir_cols   = sorted(
    [c for c in sample_df.columns if c.startswith("CIR") and c[3:].isdigit()],
    key=lambda c: int(c[3:])
)
# Show a few sample values
for col in cir_cols[:5]:
    sample_val = sample_df[col].dropna().iloc[0]
    print(f"  {col}: '{sample_val}'  → dtype={sample_df[col].dtype}")

# ── 3. CHECK KEY COLUMNS EXIST ───────────────────────────────────────────────
print("\n" + "="*65)
print("  KEY COLUMN PRESENCE CHECK")
print("="*65)

key_cols = [
    "error", "estimated_range", "actual_range",
    "fpindex", "pkindex",
    "LOS",
    "FP_power", "RX_power", "CIRpower", "std_noise",
    "fp_ampl1", "fp_ampl2", "fp_ampl3",
    "pkpathampl", "RXPACC", "LDEthreshold",
    "channel", "prf", "bitrate", "preamble",
]
for col in key_cols:
    present = col in common_columns
    if present:
        # Check if it has valid numeric values
        sample = pd.to_numeric(
            dfs[sample_loc][col].replace("-", np.nan), errors="coerce"
        )
        pct_valid = sample.notna().mean() * 100
        print(f"  {'✓' if present else '✗'} {col:<20} valid%={pct_valid:.0f}%")
    else:
        print(f"  ✗ {col:<20} MISSING")

# ── 4. PARSE CIR PROPERLY (object dtype = complex strings) ───────────────────
print("\n" + "="*65)
print("  PARSING CIR (complex strings → real + imag)")
print("="*65)

_RE_IMAG = re.compile(r'^([+-]?(?:\d+\.?\d*|\.\d+)(?:[eE][+-]?\d+)?)j$')
_RE_FULL = re.compile(
    r'^([+-]?(?:\d+\.?\d*|\.\d+)(?:[eE][+-]?\d+)?)'
    r'([+-](?:\d+\.?\d*|\.\d+)(?:[eE][+-]?\d+)?)j$'
)

def parse_complex(s):
    """Parse complex string → (real, imag). All edge cases handled."""
    try:
        s = str(s).strip().replace(" ", "")
        if s in ("nan","NaN","","−","-","None","inf"):
            return 0.0, 0.0
        s = s.replace("+-", "-")
        c = complex(s)
        return float(c.real), float(c.imag)
    except Exception:
        pass
    s2 = str(s).strip().replace(" ", "").replace("+-", "-")
    m = _RE_IMAG.match(s2)
    if m:
        return 0.0, float(m.group(1))
    m = _RE_FULL.match(s2)
    if m:
        return float(m.group(1)), float(m.group(2))
    try:
        return float(s2), 0.0
    except Exception:
        return 0.0, 0.0


def extract_cir_iq(df, cir_len=500):
    """
    Returns real (N, cir_len) and imag (N, cir_len).
    CIR centred on fpindex as in the paper (Section V).
    Since all dtypes are now consistent (object), we always parse as complex string.
    """
    all_cir_cols = sorted(
        [c for c in df.columns if c.startswith("CIR") and c[3:].isdigit()],
        key=lambda c: int(c[3:])
    )
    n_total = len(all_cir_cols)
    n_rows  = len(df)
    half    = cir_len // 2

    # fpindex: centre point reported by DW1000
    if "fpindex" in df.columns:
        fpidx = pd.to_numeric(df["fpindex"], errors="coerce").fillna(n_total//2).astype(int).values
    else:
        fpidx = np.full(n_rows, n_total // 2, dtype=int)

    # Parse ALL CIR columns at once (real + imag)
    raw_real = np.zeros((n_rows, n_total), dtype=np.float32)
    raw_imag = np.zeros((n_rows, n_total), dtype=np.float32)

    for ci, col in enumerate(all_cir_cols):
        parsed       = [parse_complex(v) for v in df[col].values]
        raw_real[:, ci] = [p[0] for p in parsed]
        raw_imag[:, ci] = [p[1] for p in parsed]

    # Centre each row around its fpindex
    out_real = np.zeros((n_rows, cir_len), dtype=np.float32)
    out_imag = np.zeros((n_rows, cir_len), dtype=np.float32)

    for i in range(n_rows):
        fp    = np.clip(int(fpidx[i]), 0, n_total - 1)
        start = max(0, fp - half)
        end   = min(n_total, fp + half)
        seg_r = raw_real[i, start:end]
        seg_im = raw_imag[i, start:end]
        L = len(seg_r)
        o_start = half - (fp - start)
        o_start = max(0, o_start)
        o_end   = min(cir_len, o_start + L)
        take    = o_end - o_start
        out_real[i, o_start:o_end] = seg_r[:take]
        out_imag[i, o_start:o_end] = seg_im[:take]

    return out_real, out_imag   # each (N, cir_len)


# Test on one location
test_loc = list(dfs.keys())[0]
R_test, I_test = extract_cir_iq(dfs[test_loc], cir_len=500)
has_imag = (I_test != 0).any(axis=1).mean() * 100
print(f"  Loc {test_loc}: parsed shape={R_test.shape} | "
      f"rows with non-zero imag: {has_imag:.1f}%")
print(f"  Real range : [{R_test.min():.4f}, {R_test.max():.4f}]")
print(f"  Imag range : [{I_test.min():.4f}, {I_test.max():.4f}]")

# ── 5. BUILD FULL DATASET (all locations) ────────────────────────────────────
print("\n" + "="*65)
print("  BUILDING FULL DATASET (all locations)")
print("="*65)

CIR_LEN = 500
all_real, all_imag, all_error, all_loc_id = [], [], [], []

for loc, df in sorted(dfs.items()):
    R, Im = extract_cir_iq(df, cir_len=CIR_LEN)
    error = pd.to_numeric(df["error"].replace("-", np.nan),
                          errors="coerce").fillna(0).values.astype(np.float32)
    all_real.append(R)
    all_imag.append(Im)
    all_error.append(error)
    all_loc_id.extend([loc] * len(df))

X_real  = np.concatenate(all_real,  axis=0)   # (N_total, 500)
X_imag  = np.concatenate(all_imag,  axis=0)   # (N_total, 500)
y_error = np.concatenate(all_error, axis=0)   # (N_total,)
loc_ids = np.array(all_loc_id)

# Flat 1000-dim input = paper's "500 × 2"
X_flat  = np.concatenate([X_real, X_imag], axis=1)  # (N_total, 1000)

print(f"  X_flat shape : {X_flat.shape}   ← (N, 1000) = real_500 + imag_500")
print(f"  y_error shape: {y_error.shape}")
print(f"  Locations    : {sorted(np.unique(loc_ids).tolist())}")
print(f"  Total samples: {len(y_error):,}")
print(f"  |error| mean : {np.abs(y_error).mean():.2f} mm  (overall, uncorrected)")

# ── 6. TRAIN / TEST SPLIT ────────────────────────────────────────────────────
TRAIN_LOCS = [1, 2, 3, 5,6,7,8,9,10,12,13,14,15,16,17,18,19,20,21,23]
TEST_LOCS  = [4,11,22]

tr_mask = np.isin(loc_ids, TRAIN_LOCS)
te_mask = np.isin(loc_ids, TEST_LOCS)

X_train, y_train = X_flat[tr_mask], y_error[tr_mask]
X_test,  y_test  = X_flat[te_mask], y_error[te_mask]

print(f"\n  Train ({TRAIN_LOCS}): {X_train.shape[0]:,} samples | "
      f"|error| mean={np.abs(y_train).mean():.2f} mm")
print(f"  Test  ({TEST_LOCS }): {X_test.shape[0]:,} samples | "
      f"|error| mean={np.abs(y_test).mean():.2f} mm")

# ── 7. QUICK VISUALISATION ───────────────────────────────────────────────────
DARK = "#0e1117"; CARD = "#1a1f2e"; CYAN = "#00d4ff"; GREEN = "#00ff88"; ORG = "#ff6b35"

def sax(ax, title):
    ax.set_facecolor(CARD)
    ax.tick_params(colors="white", labelsize=9)
    for sp in ax.spines.values(): sp.set_edgecolor("#333")
    ax.set_title(title, color="white", fontsize=10, pad=7, fontweight="bold")
    ax.xaxis.label.set_color("white"); ax.yaxis.label.set_color("white")

fig = plt.figure(figsize=(20, 12))
fig.patch.set_facecolor(DARK)
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# (a) Sample CIR real + imag from one row
ax0 = fig.add_subplot(gs[0, :2]); sax(ax0, "Sample CIR — Real & Imaginary (1 measurement, Loc 1)")
sample_idx = 0
ax0.plot(X_real[sample_idx],  color=CYAN,  lw=1.2, label="Real part",  alpha=0.9)
ax0.plot(X_imag[sample_idx],  color=ORG,   lw=1.2, label="Imag part",  alpha=0.9)
ax0.plot(np.sqrt(X_real[sample_idx]**2 + X_imag[sample_idx]**2),
         color=GREEN, lw=1.5, label="|Magnitude|", alpha=0.8)
ax0.axvline(250, color="white", ls="--", lw=0.8, alpha=0.4, label="Centre (fpindex)")
ax0.set_xlabel("CIR sample index (centred on fpindex)")
ax0.set_ylabel("Amplitude")
ax0.legend(facecolor=DARK, labelcolor="white", fontsize=9)

# (b) Error distribution per location
ax1 = fig.add_subplot(gs[0, 2]); sax(ax1, "Ranging Error Distribution per Location")
for loc, df in sorted(dfs.items()):
    err = pd.to_numeric(df["error"].replace("-", np.nan), errors="coerce").fillna(0)
    ax1.scatter([loc]*len(err), err, s=0.5, alpha=0.08, color=CYAN, rasterized=True)
ax1.axhline(0, color="white", ls="--", lw=0.8)
ax1.set_xlabel("Location"); ax1.set_ylabel("Error (mm)")

# (c) Sample count per location
ax2 = fig.add_subplot(gs[1, 0]); sax(ax2, "Sample Count per Location")
locs_sorted = sorted(dfs.keys())
counts = [len(dfs[l]) for l in locs_sorted]
cols   = [GREEN if l in TRAIN_LOCS else ORG for l in locs_sorted]
ax2.bar(locs_sorted, counts, color=cols)
ax2.set_xlabel("Location"); ax2.set_ylabel("Samples")
from matplotlib.patches import Patch
ax2.legend(handles=[Patch(color=GREEN, label="Train"), Patch(color=ORG, label="Test")],
           facecolor=DARK, labelcolor="white", fontsize=8)

# (d) |Error| MAE per location
ax3 = fig.add_subplot(gs[1, 1]); sax(ax3, "|Error| MAE per Location (mm)")
maes  = []
for l in locs_sorted:
    e = pd.to_numeric(dfs[l]["error"].replace("-", np.nan), errors="coerce").fillna(0)
    maes.append(e.abs().mean())
cols = [GREEN if l in TRAIN_LOCS else ORG for l in locs_sorted]
ax3.bar(locs_sorted, maes, color=cols)
ax3.set_xlabel("Location"); ax3.set_ylabel("MAE (mm)")
ax3.axhline(np.mean(maes), color="white", ls="--", lw=0.8, label=f"Mean={np.mean(maes):.1f}mm")
ax3.legend(facecolor=DARK, labelcolor="white", fontsize=8)

# (e) CIR magnitude heatmap (first 100 samples from loc 1)
ax4 = fig.add_subplot(gs[1, 2]); sax(ax4, "CIR Magnitude Heatmap (first 100 samples, Loc 1)")
loc1_mask = loc_ids == list(sorted(dfs.keys()))[0]
mag_subset = np.sqrt(X_real[loc1_mask][:100]**2 + X_imag[loc1_mask][:100]**2)
im = ax4.imshow(mag_subset, aspect="auto", cmap="inferno", interpolation="nearest")
ax4.set_xlabel("CIR index"); ax4.set_ylabel("Sample #")
plt.colorbar(im, ax=ax4).ax.yaxis.set_tick_params(color="white")

fig.suptitle("UWB Dataset Audit — Ready for Modelling",
             color="white", fontsize=14, fontweight="bold", y=0.995)
plt.savefig("uwb_data_audit.png", dpi=150, bbox_inches="tight", facecolor=DARK)
plt.show()
print("\n✓ Saved: uwb_data_audit.png")

# ── 8. FINAL READINESS CHECKLIST ─────────────────────────────────────────────
print("\n" + "="*65)
print("  READINESS CHECKLIST")
print("="*65)
print(f"  ✓ All dtypes consistent across locations (0 conflicts)")
print(f"  ✓ CIR parsed as complex → real + imag")
print(f"  ✓ CIR centred on fpindex (500 samples)")
print(f"  ✓ X_train shape : {X_train.shape}")
print(f"  ✓ X_test  shape : {X_test.shape}")
print(f"  ✓ y_train range : [{y_train.min():.1f}, {y_train.max():.1f}] mm")
print(f"  ✓ y_test  range : [{y_test.min():.1f}, {y_test.max():.1f}] mm")
print(f"\n  NEXT STEP: pass X_train, X_test, y_train, y_test")
print(f"  into uwb_paper_replication.py (already uses same format)")
print("="*65)

# These variables are ready to use in the next cell:
print("\n  Variables exported for next cell:")
print("  X_train, X_test   — (N, 1000) float32 IQ arrays")
print("  y_train, y_test   — (N,) float32 error in mm")
print("  X_flat, y_error, loc_ids — full dataset with location labels")
print("  X_real, X_imag    — (N, 500) real and imaginary parts separately")

  DATA AUDIT
  Loc  1:   1598 rows | err mean=  +34.1  std= 279.9  |err| mean= 162.4 mm | NaN err:   0 | LOS%:   0.0 | CIR dtype: object
  Loc  2:   1250 rows | err mean=  +86.2  std= 303.3  |err| mean= 175.9 mm | NaN err:   0 | LOS%:   0.0 | CIR dtype: object
  Loc  3:   1194 rows | err mean=  +41.3  std= 234.7  |err| mean= 147.6 mm | NaN err:   0 | LOS%:   0.0 | CIR dtype: object
  Loc  4:   1423 rows | err mean=  +57.8  std= 214.3  |err| mean= 146.8 mm | NaN err:   0 | LOS%:   0.0 | CIR dtype: object
  Loc  5:   1325 rows | err mean= +204.1  std= 590.3  |err| mean= 299.8 mm | NaN err:   0 | LOS%:   0.0 | CIR dtype: object
  Loc  6:   1376 rows | err mean=  +35.0  std= 552.0  |err| mean= 218.4 mm | NaN err:   0 | LOS%:   0.0 | CIR dtype: object
  Loc  7:   1538 rows | err mean=  +79.4  std= 479.1  |err| mean= 185.6 mm | NaN err:   0 | LOS%:   0.0 | CIR dtype: object
  Loc  8:   1110 rows | err mean= +104.9  std= 303.9  |err| mean= 203.8 mm | NaN err:   0 | LOS%:   0.0 | CIR dtype: ob

  Loc 1: parsed shape=(1598, 500) | rows with non-zero imag: 100.0%
  Real range : [-56.4728, 60.9262]
  Imag range : [-27.2193, 37.5408]

  BUILDING FULL DATASET (all locations)
  X_flat shape : (32276, 1000)   ← (N, 1000) = real_500 + imag_500
  y_error shape: (32276,)
  Locations    : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
  Total samples: 32,276
  |error| mean : 209.76 mm  (overall, uncorrected)

  Train ([1, 2, 3, 5, 6, 7, 8, 9, 10, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 23]): 27,904 samples | |error| mean=218.02 mm
  Test  ([4, 11, 22]): 4,372 samples | |error| mean=157.01 mm

✓ Saved: uwb_data_audit.png

  READINESS CHECKLIST
  ✓ All dtypes consistent across locations (0 conflicts)
  ✓ CIR parsed as complex → real + imag
  ✓ CIR centred on fpindex (500 samples)
  ✓ X_train shape : (27904, 1000)
  ✓ X_test  shape : (4372, 1000)
  ✓ y_train range : [-435.0, 11434.0] mm
  ✓ y_test  range : [-335.0, 1196.0] mm

  NEXT STEP: pass X_train,

In [2]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║  UWB RANGING ERROR CORRECTION – DUAL-LOSS SEMI-SUPERVISED AUTOENCODER      ║
║  Replication of Fontaine et al. (2020), IEEE Access 10.1109/ACCESS.2020    ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  Paper targets (Table 3)                                                   ║
║    No correction  : 214.7 mm                                               ║
║    DNN (baseline) :  75.7 mm                                               ║
║    CNN            :  60.6 mm                                               ║
║    Pre-trained AE :  69.4 mm                                               ║
║    Dual-loss AEP  :  58.6 mm  ← our target                                ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  USAGE                                                                     ║
║    Option A – Standalone (needs CSV files in same dir):                    ║
║      python uwb_dual_loss_aep.py                                           ║
║                                                                            ║
║    Option B – After your audit cell (variables already in memory):         ║
║      Just run this script/cell; it auto-detects X_train etc.              ║
║                                                                            ║
║  REQUIREMENTS                                                              ║
║    pip install torch numpy pandas matplotlib                               ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""

# ─────────────────────────────────────────────────────────────────────────────
# 0.  IMPORTS & REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────────────────────
import os, random, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, random_split

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU  : {torch.cuda.get_device_name(0)}")

# ─────────────────────────────────────────────────────────────────────────────
# 1.  DATA LOADING
#
#  Paper split (Section V):
#    Train : 20 locations  → locs 1–3, 5–10, 12–21, 23   (28,473 measurements)
#    Test  : 3 held-out    → locs 4, 11, 22              (generalisation test)
#
#  CIR format:
#    DW1000 provides 1016 complex taps (PRF=64 MHz).
#    The paper centres 500 taps around fpindex so the first-path peak is
#    always at tap 250. This gives (N, 500) complex = (N, 1000) real/imag.
# ─────────────────────────────────────────────────────────────────────────────

TRAIN_LOCS = [1,2,3,5,6,7,8,9,10,12,13,14,15,16,17,18,19,20,21,23]
TEST_LOCS  = [4, 11, 22]


def parse_complex_col(series):
    """Convert '0.12+0.01j' string series → complex128 array."""
    def _c(v):
        try:
            return complex(str(v).replace(" ", ""))
        except Exception:
            return 0 + 0j
    return np.array([_c(v) for v in series], dtype=np.complex128)


def load_location(csv_path: Path) -> tuple:
    """
    Returns X (N, 1000) float32 and y (N,) float32 for one CSV file.
    CIR columns CIR0…CIR499 are parsed as complex, centred on fpindex,
    then split into real (first 500 cols) and imag (last 500 cols).
    """
    df = pd.read_csv(csv_path)

    cir_cols = [f"CIR{i}" for i in range(500)]
    fpidx    = df["fpindex"].values.astype(int)

    # Parse 500 complex CIR taps → (N, 500) complex
    cir = np.stack([parse_complex_col(df[c]) for c in cir_cols], axis=1)

    # Centre each row so that fpindex lands at position 250
    centered = np.zeros_like(cir)
    for i, fi in enumerate(fpidx):
        shift = 250 - int(fi)
        centered[i] = np.roll(cir[i], shift)

    # Split real / imag → (N, 1000)
    X = np.concatenate([centered.real, centered.imag], axis=1).astype(np.float32)
    y = df["error"].values.astype(np.float32)
    return X, y


def load_split(data_dir="."):
    """Load and split all CSVs into train/test arrays."""
    data_dir = Path(data_dir)
    X_tr_parts, y_tr_parts = [], []
    X_te_parts, y_te_parts = [], []

    for loc in range(1, 24):
        fp = data_dir / f"exportloc{loc}IQ.csv"
        if not fp.exists():
            print(f"  WARNING: {fp} not found – skipping.")
            continue
        X, y = load_location(fp)
        if loc in TRAIN_LOCS:
            X_tr_parts.append(X); y_tr_parts.append(y)
        else:
            X_te_parts.append(X); y_te_parts.append(y)

    X_train = np.vstack(X_tr_parts);  y_train = np.concatenate(y_tr_parts)
    X_test  = np.vstack(X_te_parts);  y_test  = np.concatenate(y_te_parts)
    return X_train, X_test, y_train, y_test


def get_data(data_dir="."):
    """
    Tries to find X_train / y_train in the calling frame (notebook mode),
    falls back to loading CSVs from disk.
    """
    import inspect
    caller_locals = inspect.currentframe().f_back.f_locals
    if all(k in caller_locals for k in ("X_train", "X_test", "y_train", "y_test")):
        print("✓ Using variables already in memory (notebook mode).")
        return (caller_locals["X_train"].astype(np.float32),
                caller_locals["X_test"].astype(np.float32),
                caller_locals["y_train"].astype(np.float32),
                caller_locals["y_test"].astype(np.float32))
    print(f"Loading CSVs from: {Path(data_dir).resolve()}")
    return load_split(data_dir)


X_train_np, X_test_np, y_train_np, y_test_np = get_data(".")

print(f"\nData shapes:")
print(f"  X_train : {X_train_np.shape}   y_train : {y_train_np.shape}")
print(f"  X_test  : {X_test_np.shape}    y_test  : {y_test_np.shape}")
print(f"  Train |error| MAE : {np.abs(y_train_np).mean():.1f} mm")
print(f"  Test  |error| MAE : {np.abs(y_test_np).mean():.1f} mm")

# ─────────────────────────────────────────────────────────────────────────────
# 2.  NORMALISATION  (z-score per feature, fit on train only)
# ─────────────────────────────────────────────────────────────────────────────
X_mean = X_train_np.mean(axis=0, keepdims=True)
X_std  = X_train_np.std(axis=0, keepdims=True) + 1e-8

X_train_n = (X_train_np - X_mean) / X_std
X_test_n  = (X_test_np  - X_mean) / X_std

# Convert to tensors
X_tr_t = torch.tensor(X_train_n, dtype=torch.float32)
X_te_t = torch.tensor(X_test_n,  dtype=torch.float32)
y_tr_t = torch.tensor(y_train_np, dtype=torch.float32)
y_te_t = torch.tensor(y_test_np,  dtype=torch.float32)

# ─────────────────────────────────────────────────────────────────────────────
# 3.  HELPER – reshape (N, 1000) flat → (N, 2, 500) for Conv1d
# ─────────────────────────────────────────────────────────────────────────────
def to_cir(x_flat: torch.Tensor) -> torch.Tensor:
    """
    x_flat : (N, 1000)  [real_500 | imag_500]
    returns : (N, 2, 500)  channel-0=real, channel-1=imag
    """
    real = x_flat[:, :500].unsqueeze(1)   # (N,1,500)
    imag = x_flat[:, 500:].unsqueeze(1)   # (N,1,500)
    return torch.cat([real, imag], dim=1) # (N,2,500)


def add_noise(x_flat: torch.Tensor, std: float = 0.1) -> torch.Tensor:
    """Additive Gaussian noise for denoising AE training."""
    return x_flat + torch.randn_like(x_flat) * std

# ─────────────────────────────────────────────────────────────────────────────
# 4.  MODEL ARCHITECTURE  (Table 2 of the paper)
#
#  ┌─────────────────────────────────────────────────────────────────────┐
#  │  INPUT  (N, 2, 500)  – 2-channel IQ CIR, 500 time taps            │
#  │                                                                     │
#  │  ENCODER                                                            │
#  │    Conv1d(2→32, k=4) ReLU  → (N, 32, 500)                         │
#  │    MaxPool1d(4)             → (N, 32, 125)                         │
#  │    Conv1d(32→64, k=3) ReLU → (N, 64, 125)                         │
#  │    Conv1d(64→64, k=3) ReLU → (N, 64, 125)                         │
#  │    Conv1d(64→1,  k=2) ReLU → latent h (N, 1, 125)                 │
#  │                                                                     │
#  │  DECODER  (reconstruction branch)                                   │
#  │    Conv1d(1→64,  k=3) ReLU                                         │
#  │    Conv1d(64→64, k=3) ReLU                                         │
#  │    Upsample(×4, linear)    → (N, 64, 500)                         │
#  │    Conv1d(64→32, k=4) ReLU                                         │
#  │    Conv1d(32→2,  k=1) ReLU → (N, 2, 500)                          │
#  │                                                                     │
#  │  PREDICTOR  (prediction branch, shared encoder latent h)           │
#  │    Conv1d(1→16, k=4) ReLU                                          │
#  │    MaxPool1d(2)                                                     │
#  │    Dropout(0.25)                                                    │
#  │    AdaptiveAvgPool → (N,16,500) → Flatten → 8000               │
#  │    Dense(75) + BN + ReLU                                            │
#  │    Dense(75) + BN + ReLU                                            │
#  │    Dropout(0.15)                                                    │
#  │    Dense(25) + BN + ReLU                                            │
#  │    Dense(8)  + ReLU                                                 │
#  │    Dense(1)  → predicted ranging error ε̂ᵢ (mm)                   │
#  └─────────────────────────────────────────────────────────────────────┘
# ─────────────────────────────────────────────────────────────────────────────

class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv1d(2,  32, kernel_size=4, padding=2)
        self.pool1 = nn.MaxPool1d(4)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv1d(64, 64, kernel_size=3, padding=1)
        self.conv4 = nn.Conv1d(64,  1, kernel_size=2, padding=1)

    def forward(self, x):             # x  : (N, 2, 500)
        x = F.relu(self.conv1(x))     #      (N, 32, 503)
        x = x[:, :, :500]            #      (N, 32, 500) – trim padding
        x = self.pool1(x)            #      (N, 32, 125)
        x = F.relu(self.conv2(x))    #      (N, 64, 125)
        x = F.relu(self.conv3(x))    #      (N, 64, 125)
        x = F.relu(self.conv4(x))    #      (N,  1, 126)
        x = x[:, :, :125]           #      (N,  1, 125) ← latent h
        return x


class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv1d( 1, 64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(64, 64, kernel_size=3, padding=1)
        self.up    = nn.Upsample(scale_factor=4, mode="linear", align_corners=False)
        self.conv3 = nn.Conv1d(64, 32, kernel_size=4, padding=2)
        self.conv4 = nn.Conv1d(32,  2, kernel_size=1)

    def forward(self, h):             # h  : (N,  1, 125)
        x = F.relu(self.conv1(h))    #      (N, 64, 125)
        x = F.relu(self.conv2(x))    #      (N, 64, 125)
        x = self.up(x)               #      (N, 64, 500)
        x = F.relu(self.conv3(x))    #      (N, 32, 503)
        x = x[:, :, :500]           #      (N, 32, 500)
        x = F.relu(self.conv4(x))   #      (N,  2, 500) ← reconstructed CIR
        return x


class Predictor(nn.Module):
    """CNN predictor attached to the shared latent h."""
    def __init__(self):
        super().__init__()
        self.conv1  = nn.Conv1d(1, 16, kernel_size=4, padding=2)
        self.pool1  = nn.MaxPool1d(2)
        self.drop1  = nn.Dropout(0.25)
        self.conv2  = nn.Conv1d(16, 16, kernel_size=3, padding=1)
        # Adaptive pool → 500 → flatten → 8000  (matches paper's 8000 FC input)
        self.adapt  = nn.AdaptiveAvgPool1d(500)
        self.flat   = nn.Flatten()

        self.fc1    = nn.Linear(8000, 75); self.bn1 = nn.BatchNorm1d(75)
        self.fc2    = nn.Linear(75,   75); self.bn2 = nn.BatchNorm1d(75)
        self.drop2  = nn.Dropout(0.15)
        self.fc3    = nn.Linear(75,   25); self.bn3 = nn.BatchNorm1d(25)
        self.fc4    = nn.Linear(25,    8)
        self.fc5    = nn.Linear( 8,    1)

    def forward(self, h):              # h : (N, 1, 125)
        x = F.relu(self.conv1(h))     #     (N, 16, ~127)
        x = self.pool1(x)             #     (N, 16, ~63)
        x = self.drop1(x)
        x = F.relu(self.conv2(x))
        x = self.adapt(x)             #     (N, 16, 500)
        x = self.flat(x)              #     (N, 8000)
        x = F.relu(self.bn1(self.fc1(x)))
        x = F.relu(self.bn2(self.fc2(x)))
        x = self.drop2(x)
        x = F.relu(self.bn3(self.fc3(x)))
        x = F.relu(self.fc4(x))
        x = self.fc5(x)               #     (N, 1)
        return x.squeeze(1)           #     (N,)


class DualLossAEP(nn.Module):
    """
    Semi-supervised Autoencoder Predictor (AEP).

    Training loss:
        L_AEP = W_AE * L_AE  +  W_CNN * L_CNN
        L_AE  = MAE(decoder(encoder(x_noisy)), x_clean)
        L_CNN = MAE(predictor(encoder(x_noisy)), ε_true)

    Inference:
        corrected_range = estimated_range − predictor(encoder(CIR))
    """
    def __init__(self):
        super().__init__()
        self.encoder   = Encoder()
        self.decoder   = Decoder()
        self.predictor = Predictor()

    def forward(self, x):
        h     = self.encoder(x)
        recon = self.decoder(h)
        pred  = self.predictor(h)
        return recon, pred

    def predict(self, x):
        """Inference-only: return predicted ranging error."""
        with torch.no_grad():
            h = self.encoder(x)
            return self.predictor(h)


class DNN(nn.Module):
    """DNN baseline – Table 2, [16]. Input=500×2=1000 floats."""
    def __init__(self, input_dim=1000):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(input_dim, 50), nn.ReLU(),
            nn.Linear(50, 50),        nn.ReLU(),
            nn.Linear(50, 50),        nn.ReLU(),
            nn.Linear(50,  1)
        )

    def forward(self, x):
        # x can be (N,2,500) or (N,1000) – flatten either way
        if x.dim() == 3:
            x = x.reshape(x.size(0), -1)
        return self.net(x).squeeze(1)


# Print parameter counts
aep = DualLossAEP()
dnn_m = DNN()
def count_params(m): return sum(p.numel() for p in m.parameters())
print(f"\nModel parameters:")
print(f"  Dual-loss AEP : {count_params(aep):,}  (paper: 32,019)")
print(f"  DNN baseline  : {count_params(dnn_m):,}  (paper: 30,800)")

# ─────────────────────────────────────────────────────────────────────────────
# 5.  TRAINING CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

# ── Loss weights (Equation 5 in paper) ───────────────────────────────────────
#   Higher W_CNN → more weight on prediction → better ranging correction.
#   Start 0.3/0.7 (balanced) or go aggressive 0.1/0.9 if AE reconstructs well.
W_AE  = 0.3
W_CNN = 0.7

# ── Denoising noise level ─────────────────────────────────────────────────────
NOISE_STD = 0.1    # σ for Gaussian noise added to CIR (normalised scale)

# ── Optimiser ─────────────────────────────────────────────────────────────────
LR          = 1e-3
WEIGHT_DECAY= 1e-5
MAX_EPOCHS  = 500
PATIENCE    = 25   # early stopping patience (paper: 25 epochs)

# ── Batch & DataLoader ────────────────────────────────────────────────────────
BATCH_SIZE  = 64

# ─────────────────────────────────────────────────────────────────────────────
# 6.  DATASET SPLIT & LOADERS
#     90% train / 10% validation from the training locations.
# ─────────────────────────────────────────────────────────────────────────────
full_ds = TensorDataset(X_tr_t, y_tr_t)
n_val   = int(0.10 * len(full_ds))
n_tr    = len(full_ds) - n_val
tr_ds, val_ds = random_split(full_ds, [n_tr, n_val],
                              generator=torch.Generator().manual_seed(SEED))

tr_loader  = DataLoader(tr_ds,  batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=0, pin_memory=(DEVICE.type=="cuda"))
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=0, pin_memory=(DEVICE.type=="cuda"))
te_loader  = DataLoader(TensorDataset(X_te_t, y_te_t),
                        batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=0, pin_memory=(DEVICE.type=="cuda"))

print(f"\nDataset splits:")
print(f"  Train   : {n_tr:,}   Val : {n_val:,}   Test : {len(y_te_t):,}")

# ─────────────────────────────────────────────────────────────────────────────
# 7.  TRAINING FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────
mae_fn = nn.L1Loss()


def train_aep_epoch(model, loader, optimizer):
    model.train()
    tot_ae = tot_cnn = tot_loss = 0.0
    for Xb, yb in loader:
        Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
        X_clean = to_cir(Xb)
        X_noisy = to_cir(add_noise(Xb, NOISE_STD))

        optimizer.zero_grad(set_to_none=True)
        recon, pred = model(X_noisy)            # predict from noisy CIR
        l_ae  = mae_fn(recon, X_clean)          # reconstruct clean CIR
        l_cnn = mae_fn(pred, yb)
        loss  = W_AE * l_ae + W_CNN * l_cnn
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        n = yb.size(0)
        tot_ae   += l_ae.item()  * n
        tot_cnn  += l_cnn.item() * n
        tot_loss += loss.item()  * n
    N = len(loader.dataset)
    return tot_ae/N, tot_cnn/N, tot_loss/N


@torch.no_grad()
def eval_aep(model, loader):
    model.eval()
    tot_ae = tot_cnn = tot_loss = 0.0
    for Xb, yb in loader:
        Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
        X_clean = to_cir(Xb)
        X_noisy = to_cir(add_noise(Xb, NOISE_STD))
        recon, pred = model(X_noisy)
        l_ae  = mae_fn(recon, X_clean)
        l_cnn = mae_fn(pred, yb)
        loss  = W_AE * l_ae + W_CNN * l_cnn
        n = yb.size(0)
        tot_ae   += l_ae.item()  * n
        tot_cnn  += l_cnn.item() * n
        tot_loss += loss.item()  * n
    N = len(loader.dataset)
    return tot_ae/N, tot_cnn/N, tot_loss/N


def train_generic(model, loader_tr, loader_val, tag="model"):
    """Generic MAE regression trainer (used for DNN baseline)."""
    model.to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=10, factor=0.5, min_lr=1e-5)
    best_loss, best_state, no_imp = float("inf"), None, 0
    hist = []
    for epoch in range(1, MAX_EPOCHS+1):
        model.train()
        tr_loss = 0.0
        for Xb, yb in loader_tr:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            loss = mae_fn(model(Xb), yb)
            loss.backward(); opt.step()
            tr_loss += loss.item() * yb.size(0)
        tr_loss /= len(loader_tr.dataset)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for Xb, yb in loader_val:
                Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
                val_loss += mae_fn(model(Xb), yb).item() * yb.size(0)
        val_loss /= len(loader_val.dataset)
        sched.step(val_loss)
        hist.append((tr_loss, val_loss))

        if val_loss < best_loss:
            best_loss  = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_imp     = 0
        else:
            no_imp += 1
        if no_imp >= PATIENCE:
            print(f"  [{tag}] Early stop at epoch {epoch:4d} | best val={best_loss:.1f} mm")
            break
        if epoch % 50 == 0:
            print(f"  [{tag}] Ep {epoch:4d} | tr={tr_loss:.1f}  val={val_loss:.1f} mm")

    model.load_state_dict(best_state)
    return model, hist


@torch.no_grad()
def get_predictions(model, loader, is_aep=True):
    model.eval()
    preds, trues = [], []
    for Xb, yb in loader:
        Xb = Xb.to(DEVICE)
        if is_aep:
            _, pred = model(to_cir(Xb))
        else:
            pred = model(Xb)
        preds.append(pred.cpu()); trues.append(yb)
    return torch.cat(preds).numpy(), torch.cat(trues).numpy()


# ─────────────────────────────────────────────────────────────────────────────
# 8.  TRAIN DUAL-LOSS AEP
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "═"*65)
print("  DUAL-LOSS SEMI-SUPERVISED AEP")
print("═"*65)
print(f"  W_AE={W_AE}  W_CNN={W_CNN}  noise_std={NOISE_STD}  "
      f"batch={BATCH_SIZE}  lr={LR}")

aep = DualLossAEP().to(DEVICE)
aep_opt   = torch.optim.Adam(aep.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
aep_sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
                aep_opt, patience=10, factor=0.5, min_lr=1e-5)

aep_history = {"tr_ae":[], "tr_cnn":[], "tr_tot":[],
               "val_ae":[], "val_cnn":[], "val_tot":[]}

best_aep_loss  = float("inf")
best_aep_state = None
aep_no_imp     = 0

for epoch in range(1, MAX_EPOCHS+1):
    tr_ae,  tr_cnn,  tr_tot  = train_aep_epoch(aep, tr_loader, aep_opt)
    val_ae, val_cnn, val_tot = eval_aep(aep, val_loader)
    aep_sched.step(val_tot)

    aep_history["tr_ae"].append(tr_ae);    aep_history["tr_cnn"].append(tr_cnn)
    aep_history["tr_tot"].append(tr_tot)
    aep_history["val_ae"].append(val_ae);  aep_history["val_cnn"].append(val_cnn)
    aep_history["val_tot"].append(val_tot)

    if val_tot < best_aep_loss:
        best_aep_loss  = val_tot
        best_aep_state = {k: v.cpu().clone() for k, v in aep.state_dict().items()}
        aep_no_imp     = 0
    else:
        aep_no_imp += 1

    if epoch % 10 == 0 or epoch == 1:
        lr_now = aep_opt.param_groups[0]["lr"]
        print(f"  Ep {epoch:4d} | tr_cnn={tr_cnn:6.1f} mm  val_cnn={val_cnn:6.1f} mm | "
              f"tr_ae={tr_ae:.4f}  val_ae={val_ae:.4f} | lr={lr_now:.6f} | patience={aep_no_imp}")

    if aep_no_imp >= PATIENCE:
        print(f"\n  ⏹  Early stop at epoch {epoch}  (best val_tot={best_aep_loss:.4f})")
        break

aep.load_state_dict(best_aep_state)
aep_preds, aep_true = get_predictions(aep, te_loader, is_aep=True)

# ─────────────────────────────────────────────────────────────────────────────
# 9.  TRAIN DNN BASELINE
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "─"*65)
print("  DNN BASELINE  [Tiemann et al., 2017]")
print("─"*65)
dnn_m, dnn_hist = train_generic(DNN(), tr_loader, val_loader, tag="DNN")
dnn_preds, dnn_true = get_predictions(dnn_m, te_loader, is_aep=False)

# ─────────────────────────────────────────────────────────────────────────────
# 10. METRICS
# ─────────────────────────────────────────────────────────────────────────────
def mae(a, b=None):
    return float(np.abs(a if b is None else (a - b)).mean())

raw_mae_val     = mae(aep_true)                  # uncorrected |error|
aep_corr_mae    = mae(aep_true, aep_preds)       # after AEP correction
dnn_corr_mae    = mae(dnn_true, dnn_preds)       # after DNN correction

# 95th-percentile errors
aep_p95 = float(np.percentile(np.abs(aep_true - aep_preds), 95))
raw_p95 = float(np.percentile(np.abs(aep_true), 95))

print("\n" + "═"*65)
print("  FINAL RESULTS  (test locations 4, 11, 22)")
print("═"*65)
print(f"  {'Approach':<22} {'MAE (mm)':>10}  {'Paper (mm)':>12}")
print(f"  {'-'*46}")
print(f"  {'No correction':<22} {raw_mae_val:>10.1f}  {'214.7':>12}")
print(f"  {'DNN baseline':<22} {dnn_corr_mae:>10.1f}  {'75.7':>12}")
print(f"  {'Dual-loss AEP':<22} {aep_corr_mae:>10.1f}  {'58.6':>12}")
print(f"  {'-'*46}")
print(f"  AEP 95th-percentile residual : {aep_p95:.1f} mm  (paper: 350 mm)")
print(f"  Raw 95th-percentile error    : {raw_p95:.1f} mm  (paper: 700 mm)")
print()

# Per-location breakdown
loc_sizes   = {4: 1423, 11: 1416, 22: 1533}
loc_offsets = {}
offset = 0
for loc in [4, 11, 22]:
    loc_offsets[loc] = (offset, offset + loc_sizes[loc])
    offset += loc_sizes[loc]

print(f"  Per-location:")
for loc in [4, 11, 22]:
    s, e = loc_offsets[loc]
    r    = mae(aep_true[s:e])
    c    = mae(aep_true[s:e], aep_preds[s:e])
    print(f"    Loc {loc:2d}  raw={r:.1f} mm  → corrected={c:.1f} mm  "
          f"(Δ={r-c:+.1f} mm)")

# ─────────────────────────────────────────────────────────────────────────────
# 11. SAVE MODEL CHECKPOINT
# ─────────────────────────────────────────────────────────────────────────────
ckpt_path = Path("/mnt/user-data/outputs/dual_loss_aep.pt")
ckpt_path.parent.mkdir(parents=True, exist_ok=True)
torch.save({
    "epoch":        len(aep_history["val_cnn"]),
    "model_state":  best_aep_state,
    "X_mean":       X_mean,
    "X_std":        X_std,
    "config": {
        "W_AE": W_AE, "W_CNN": W_CNN, "NOISE_STD": NOISE_STD,
        "LR": LR, "BATCH_SIZE": BATCH_SIZE,
    },
    "results": {
        "raw_mae":     raw_mae_val,
        "aep_mae":     aep_corr_mae,
        "dnn_mae":     dnn_corr_mae,
    }
}, ckpt_path)
print(f"\n✓ Checkpoint saved: {ckpt_path}")

# ─────────────────────────────────────────────────────────────────────────────
# 12. PLOTS
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("UWB Ranging Error Correction – Dual-Loss AEP", fontsize=13, fontweight="bold")

# (a) Training curves
ax = axes[0]
ax.plot(aep_history["tr_cnn"],  color="steelblue",  alpha=0.6, label="Train CNN loss")
ax.plot(aep_history["val_cnn"], color="steelblue",  linewidth=2, label="Val CNN loss")
ax.plot(aep_history["tr_ae"],   color="darkorange", alpha=0.6, label="Train AE loss")
ax.plot(aep_history["val_ae"],  color="darkorange", linewidth=2, label="Val AE loss")
ax.axhline(58.6, color="red", linestyle="--", linewidth=1, label="Paper target (58.6 mm)")
ax.set_xlabel("Epoch"); ax.set_ylabel("MAE")
ax.set_title("AEP Training Curves"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# (b) CDF (mirrors Figure 6 of the paper)
ax = axes[1]
bins = np.linspace(0, 1100, 500)
for errors, label, color, ls in [
    (np.abs(aep_true),             "Uncorrected",       "gray",       "-"),
    (np.abs(aep_true - dnn_preds), f"DNN ({dnn_corr_mae:.0f} mm)",  "steelblue",  "--"),
    (np.abs(aep_true - aep_preds), f"AEP ({aep_corr_mae:.0f} mm)",  "darkorange", "-"),
]:
    sorted_e = np.sort(errors)
    cdf      = np.arange(1, len(sorted_e)+1) / len(sorted_e)
    ax.plot(sorted_e, cdf, ls, color=color, linewidth=1.8, label=label)

ax.axhline(0.95, color="black", linestyle=":", linewidth=1, label="95th pct")
ax.axvline(350,  color="red",   linestyle=":", linewidth=1, alpha=0.5, label="Paper 95% AEP")
ax.set_xlabel("Absolute ranging error (mm)"); ax.set_ylabel("CDF")
ax.set_title("CDF – Generalisation (unseen locs 4, 11, 22)")
ax.legend(fontsize=8); ax.grid(True, alpha=0.3); ax.set_xlim(0, 1000)

# (c) Scatter: predicted vs true error
ax = axes[2]
lim = max(np.abs(aep_true).max(), np.abs(aep_preds).max()) * 1.05
ax.scatter(aep_true, aep_preds, s=2, alpha=0.25, color="steelblue")
ax.plot([-lim, lim], [-lim, lim], "r--", linewidth=1.5, label="Perfect prediction")
ax.set_xlabel("True ranging error (mm)"); ax.set_ylabel("Predicted error (mm)")
ax.set_title("AEP: Prediction vs Ground Truth"); ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plot_path = Path("/mnt/user-data/outputs/dual_loss_results.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
print(f"✓ Plot saved: {plot_path}")

# ─────────────────────────────────────────────────────────────────────────────
# 13. INFERENCE EXAMPLE  (how to use the saved model)
# ─────────────────────────────────────────────────────────────────────────────
print("""
─────────────────────────────────────────────────────────────────
INFERENCE EXAMPLE (how to correct a new UWB range measurement)
─────────────────────────────────────────────────────────────────
# Load checkpoint
ckpt     = torch.load('dual_loss_aep.pt')
model    = DualLossAEP()
model.load_state_dict(ckpt['model_state'])
model.eval()

# Your new measurement: CIR as (1, 1000) array [real_500 | imag_500]
# centred on fpindex, normalised with saved mean/std
X_new    = (raw_cir - ckpt['X_mean']) / ckpt['X_std']
X_tensor = torch.tensor(X_new, dtype=torch.float32)

with torch.no_grad():
    predicted_error = model.predict(to_cir(X_tensor))  # mm

corrected_range = estimated_range - predicted_error.item()
─────────────────────────────────────────────────────────────────
""")

Device : cuda
  GPU  : Tesla T4
Loading CSVs from: /teamspace/studios/this_studio/newdatasets



Data shapes:
  X_train : (27904, 1000)   y_train : (27904,)
  X_test  : (4372, 1000)    y_test  : (4372,)
  Train |error| MAE : 218.0 mm
  Test  |error| MAE : 157.0 mm

Model parameters:
  Dual-loss AEP : 648,981  (paper: 32,019)
  DNN baseline  : 55,201  (paper: 30,800)

Dataset splits:
  Train   : 25,114   Val : 2,790   Test : 4,372

═════════════════════════════════════════════════════════════════
  DUAL-LOSS SEMI-SUPERVISED AEP
═════════════════════════════════════════════════════════════════
  W_AE=0.3  W_CNN=0.7  noise_std=0.1  batch=64  lr=0.001
  Ep    1 | tr_cnn= 216.1 mm  val_cnn= 209.9 mm | tr_ae=0.6717  val_ae=0.6783 | lr=0.001000 | patience=0
  Ep   10 | tr_cnn= 199.2 mm  val_cnn= 192.1 mm | tr_ae=0.6606  val_ae=0.6741 | lr=0.001000 | patience=0
  Ep   20 | tr_cnn= 196.7 mm  val_cnn= 191.4 mm | tr_ae=0.6669  val_ae=0.6769 | lr=0.001000 | patience=0
  Ep   30 | tr_cnn= 194.6 mm  val_cnn= 193.3 mm | tr_ae=0.6700  val_ae=0.6812 | lr=0.001000 | patience=10
  Ep   40 | tr_cnn=

PermissionError: [Errno 13] Permission denied: '/mnt/user-data'

In [3]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║  UWB RANGING ERROR CORRECTION – FULL PAPER REPLICATION                     ║
║  Fontaine et al. (2020), IEEE Access 10.1109/ACCESS.2020.3012822           ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  TWO VALIDATION MODES (both run automatically)                              ║
║                                                                             ║
║  MODE A – Combined Random Split  (Table 3 of paper)                        ║
║    All 32 k samples pooled → 80% train / 20% test (random, stratified)     ║
║    Paper targets:                                                           ║
║      No correction  : 214.7 mm                                             ║
║      DNN            :  75.7 mm                                             ║
║      CNN            :  60.6 mm                                             ║
║      Pre-trained AE :  69.4 mm                                             ║
║      Dual-loss AEP  :  58.6 mm  ← closest to paper with this mode         ║
║                                                                             ║
║  MODE B – Leave-Location-Out  (Figure 6 of paper)                          ║
║    Train: locs 1-3,5-10,12-21,23   Test: locs 4,11,22 (never seen)        ║
║    Paper CDF targets (95th pct):                                            ║
║      Uncorrected    : ~700 mm                                              ║
║      DNN corrected  : ~600 mm                                              ║
║      AEP corrected  : ~350 mm                                              ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  REQUIREMENTS: pip install torch numpy pandas matplotlib scikit-learn      ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""

# ─────────────────────────────────────────────────────────────────────────────
# 0.  IMPORTS & GLOBAL SEED
# ─────────────────────────────────────────────────────────────────────────────
import os, random, warnings, time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from copy import deepcopy

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, random_split

warnings.filterwarnings("ignore")

SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"{'='*65}")
print(f"  UWB Dual-Loss AEP – Full Paper Replication")
print(f"  Device : {DEVICE}" + (f"  ({torch.cuda.get_device_name(0)})" if DEVICE.type=="cuda" else ""))
print(f"{'='*65}\n")

# ─────────────────────────────────────────────────────────────────────────────
# 1.  DATA LOADING
# ─────────────────────────────────────────────────────────────────────────────

TRAIN_LOCS = [1,2,3,5,6,7,8,9,10,12,13,14,15,16,17,18,19,20,21,23]
TEST_LOCS  = [4, 11, 22]
ALL_LOCS   = TRAIN_LOCS + TEST_LOCS   # all 23 locations


def parse_complex_col(series):
    def _c(v):
        try:    return complex(str(v).replace(" ", ""))
        except: return 0+0j
    return np.array([_c(v) for v in series], dtype=np.complex64)


def load_location(csv_path: Path):
    """
    Returns X (N, 1000) float32 and y (N,) float32 for one location CSV.
    CIR0-499 are complex strings; centred on fpindex so peak lands at tap 250.
    Output columns: [real_0..499 | imag_0..499]
    """
    df       = pd.read_csv(csv_path)
    cir_cols = [f"CIR{i}" for i in range(500)]
    fpidx    = df["fpindex"].values.astype(int)

    cir = np.stack([parse_complex_col(df[c]) for c in cir_cols], axis=1)  # (N,500) complex

    centered = np.zeros_like(cir)
    for i, fi in enumerate(fpidx):
        centered[i] = np.roll(cir[i], 250 - int(fi))

    X = np.concatenate([centered.real, centered.imag], axis=1).astype(np.float32)
    y = df["error"].values.astype(np.float32)
    return X, y


def load_all(data_dir="."):
    """Load all 23 location CSVs.  Returns dicts {loc: (X, y)}."""
    data_dir = Path(data_dir)
    data = {}
    for loc in sorted(ALL_LOCS):
        fp = data_dir / f"exportloc{loc}IQ.csv"
        if fp.exists():
            data[loc] = load_location(fp)
        else:
            print(f"  ⚠  Missing: {fp}")
    print(f"  Loaded {len(data)} / 23 locations.")
    return data


def make_full_arrays(data):
    """Stack all locations into (X_full, y_full, loc_ids)."""
    Xs, ys, ids = [], [], []
    for loc in sorted(data.keys()):
        X, y = data[loc]
        Xs.append(X); ys.append(y)
        ids.extend([loc] * len(y))
    return np.vstack(Xs), np.concatenate(ys), np.array(ids)


# Try notebook memory first, then disk
import inspect
_caller = inspect.currentframe().f_back
if _caller and all(k in _caller.f_locals for k in ("X_flat","y_error","loc_ids")):
    print("✓ Using audit variables from notebook memory.")
    X_full    = _caller.f_locals["X_flat"].astype(np.float32)
    y_full    = _caller.f_locals["y_error"].astype(np.float32)
    loc_ids   = _caller.f_locals["loc_ids"]
    data_dict = {}   # not needed further
else:
    print("Loading CSVs from disk …")
    data_dict = load_all(".")
    X_full, y_full, loc_ids = make_full_arrays(data_dict)

print(f"\n  Full dataset : X={X_full.shape}  y={y_full.shape}")
print(f"  |error| mean : {np.abs(y_full).mean():.1f} mm  (paper: 214.7 mm)")


# ─────────────────────────────────────────────────────────────────────────────
# 2.  THE TWO SPLITS
# ─────────────────────────────────────────────────────────────────────────────

# ── MODE A: Combined random 80/20 split (Table 3) ───────────────────────────
rng     = np.random.default_rng(SEED)
idx_all = np.arange(len(X_full))
rng.shuffle(idx_all)
split   = int(0.80 * len(idx_all))
idx_A_tr, idx_A_te = idx_all[:split], idx_all[split:]

X_A_tr, y_A_tr = X_full[idx_A_tr], y_full[idx_A_tr]
X_A_te, y_A_te = X_full[idx_A_te], y_full[idx_A_te]
print(f"\n  MODE A (random 80/20): train={len(X_A_tr):,}  test={len(X_A_te):,}")
print(f"    Train |error| : {np.abs(y_A_tr).mean():.1f} mm")
print(f"    Test  |error| : {np.abs(y_A_te).mean():.1f} mm   ← should be ~214 mm (paper)")

# ── MODE B: Leave-location-out  (Figure 6) ──────────────────────────────────
mask_B_tr = np.isin(loc_ids, TRAIN_LOCS)
mask_B_te = np.isin(loc_ids, TEST_LOCS)
X_B_tr, y_B_tr = X_full[mask_B_tr], y_full[mask_B_tr]
X_B_te, y_B_te = X_full[mask_B_te], y_full[mask_B_te]
loc_B_te       = loc_ids[mask_B_te]
print(f"\n  MODE B (leave-loc-out): train={len(X_B_tr):,}  test={len(X_B_te):,}")
print(f"    Train locs : {TRAIN_LOCS}")
print(f"    Test  locs : {TEST_LOCS}")
print(f"    Test  |error| : {np.abs(y_B_te).mean():.1f} mm")


# ─────────────────────────────────────────────────────────────────────────────
# 3.  NORMALISATION  (fit on train of each mode separately)
# ─────────────────────────────────────────────────────────────────────────────

def normalise(X_tr, X_te):
    mu  = X_tr.mean(0, keepdims=True)
    sig = X_tr.std(0, keepdims=True) + 1e-8
    return (X_tr - mu) / sig, (X_te - mu) / sig, mu, sig


X_A_tr_n, X_A_te_n, mu_A, sig_A = normalise(X_A_tr, X_A_te)
X_B_tr_n, X_B_te_n, mu_B, sig_B = normalise(X_B_tr, X_B_te)


def to_tensors(Xtr, Xte, ytr, yte):
    return (torch.tensor(Xtr, dtype=torch.float32),
            torch.tensor(Xte, dtype=torch.float32),
            torch.tensor(ytr, dtype=torch.float32),
            torch.tensor(yte, dtype=torch.float32))


Xtr_A, Xte_A, ytr_A, yte_A = to_tensors(X_A_tr_n, X_A_te_n, y_A_tr, y_A_te)
Xtr_B, Xte_B, ytr_B, yte_B = to_tensors(X_B_tr_n, X_B_te_n, y_B_tr, y_B_te)


# ─────────────────────────────────────────────────────────────────────────────
# 4.  UTILITY – DataLoaders
# ─────────────────────────────────────────────────────────────────────────────

BATCH_SIZE  = 64
VAL_FRAC    = 0.10     # 10% of train → validation for early stopping
PIN_MEM     = (DEVICE.type == "cuda")


def make_loaders(X_tr_t, y_tr_t, X_te_t, y_te_t, seed=SEED):
    full_ds = TensorDataset(X_tr_t, y_tr_t)
    n_val   = int(VAL_FRAC * len(full_ds))
    n_tr    = len(full_ds) - n_val
    tr_ds, val_ds = random_split(full_ds, [n_tr, n_val],
                                 generator=torch.Generator().manual_seed(seed))
    tr_l  = DataLoader(tr_ds,  batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=0, pin_memory=PIN_MEM)
    val_l = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                       num_workers=0, pin_memory=PIN_MEM)
    te_l  = DataLoader(TensorDataset(X_te_t, y_te_t),
                       batch_size=BATCH_SIZE, shuffle=False,
                       num_workers=0, pin_memory=PIN_MEM)
    return tr_l, val_l, te_l, n_tr, n_val


# ─────────────────────────────────────────────────────────────────────────────
# 5.  CIR RESHAPE HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def to_cir(x):
    """(N,1000) → (N,2,500)  channel-0=real  channel-1=imag"""
    return torch.cat([x[:, :500].unsqueeze(1),
                      x[:, 500:].unsqueeze(1)], dim=1)


def add_noise(x, std=0.1):
    return x + torch.randn_like(x) * std


# ─────────────────────────────────────────────────────────────────────────────
# 6.  MODEL DEFINITIONS  (Table 2)
# ─────────────────────────────────────────────────────────────────────────────

class Encoder(nn.Module):
    """
    Conv(2→32,k=4) → Pool(4) → Conv(32→64,k=3) → Conv(64→64,k=3)
    → Conv(64→1,k=2) → latent h (N,1,125)
    """
    def __init__(self):
        super().__init__()
        self.c1 = nn.Conv1d(2,  32, 4, padding=2)
        self.p1 = nn.MaxPool1d(4)
        self.c2 = nn.Conv1d(32, 64, 3, padding=1)
        self.c3 = nn.Conv1d(64, 64, 3, padding=1)
        self.c4 = nn.Conv1d(64,  1, 2, padding=1)

    def forward(self, x):                          # (N,2,500)
        x = F.relu(self.c1(x))[:, :, :500]        # (N,32,500)
        x = self.p1(x)                             # (N,32,125)
        x = F.relu(self.c2(x))                     # (N,64,125)
        x = F.relu(self.c3(x))                     # (N,64,125)
        x = F.relu(self.c4(x))[:, :, :125]        # (N, 1,125) ← h
        return x


class Decoder(nn.Module):
    """
    Conv(1→64,k=3) → Conv(64→64,k=3) → Upsample(×4)
    → Conv(64→32,k=4) → Conv(32→2,k=1) → (N,2,500)
    """
    def __init__(self):
        super().__init__()
        self.c1 = nn.Conv1d( 1, 64, 3, padding=1)
        self.c2 = nn.Conv1d(64, 64, 3, padding=1)
        self.up = nn.Upsample(scale_factor=4, mode="linear", align_corners=False)
        self.c3 = nn.Conv1d(64, 32, 4, padding=2)
        self.c4 = nn.Conv1d(32,  2, 1)

    def forward(self, h):                          # (N,1,125)
        x = F.relu(self.c1(h))                    # (N,64,125)
        x = F.relu(self.c2(x))                    # (N,64,125)
        x = self.up(x)                            # (N,64,500)
        x = F.relu(self.c3(x))[:, :, :500]       # (N,32,500)
        return F.relu(self.c4(x))                 # (N, 2,500)


class Predictor(nn.Module):
    """
    Conv(1→16,k=4) → Pool(2) → Drop(0.25) → Conv(16→16,k=3)
    → AdaptPool(500) → Flat(8000)
    → FC(75)+BN+ReLU → FC(75)+BN+ReLU → Drop(0.15)
    → FC(25)+BN+ReLU → FC(8)+ReLU → FC(1)
    """
    def __init__(self):
        super().__init__()
        self.c1    = nn.Conv1d(1, 16, 4, padding=2)
        self.p1    = nn.MaxPool1d(2)
        self.d1    = nn.Dropout(0.25)
        self.c2    = nn.Conv1d(16, 16, 3, padding=1)
        self.ap    = nn.AdaptiveAvgPool1d(500)     # → (N,16,500) → flat=8000
        self.flat  = nn.Flatten()
        self.fc1   = nn.Linear(8000, 75);  self.bn1 = nn.BatchNorm1d(75)
        self.fc2   = nn.Linear(75,   75);  self.bn2 = nn.BatchNorm1d(75)
        self.d2    = nn.Dropout(0.15)
        self.fc3   = nn.Linear(75,   25);  self.bn3 = nn.BatchNorm1d(25)
        self.fc4   = nn.Linear(25,    8)
        self.fc5   = nn.Linear( 8,    1)

    def forward(self, h):                          # (N,1,125)
        x = F.relu(self.c1(h))
        x = self.d1(self.p1(x))
        x = F.relu(self.c2(x))
        x = self.flat(self.ap(x))                 # (N,8000)
        x = F.relu(self.bn1(self.fc1(x)))
        x = F.relu(self.bn2(self.fc2(x)))
        x = self.d2(x)
        x = F.relu(self.bn3(self.fc3(x)))
        x = F.relu(self.fc4(x))
        return self.fc5(x).squeeze(1)             # (N,)


class DualLossAEP(nn.Module):
    """Semi-supervised Autoencoder Predictor (AEP) – Algorithm 1."""
    def __init__(self):
        super().__init__()
        self.encoder   = Encoder()
        self.decoder   = Decoder()
        self.predictor = Predictor()

    def forward(self, x):
        h = self.encoder(x)
        return self.decoder(h), self.predictor(h)

    def predict(self, x_flat):
        with torch.no_grad():
            return self.predictor(self.encoder(to_cir(x_flat)))


class CNNBaseline(nn.Module):
    """
    Standalone CNN – same architecture as Predictor but takes raw CIR input.
    Conv(2→16,k=4) → Pool(2) → Drop(0.25) → Conv(16→16,k=3)
    → AdaptPool → Flat(8000) → same FC stack
    This is the 'CNN' row in Table 3 (60.6 mm).
    """
    def __init__(self):
        super().__init__()
        self.c1   = nn.Conv1d(2, 16, 4, padding=2)
        self.p1   = nn.MaxPool1d(2)
        self.d1   = nn.Dropout(0.25)
        self.c2   = nn.Conv1d(16, 16, 3, padding=1)
        self.ap   = nn.AdaptiveAvgPool1d(500)
        self.flat = nn.Flatten()
        self.fc1  = nn.Linear(8000, 75);  self.bn1 = nn.BatchNorm1d(75)
        self.fc2  = nn.Linear(75,   75);  self.bn2 = nn.BatchNorm1d(75)
        self.d2   = nn.Dropout(0.15)
        self.fc3  = nn.Linear(75,   25);  self.bn3 = nn.BatchNorm1d(25)
        self.fc4  = nn.Linear(25,    8)
        self.fc5  = nn.Linear( 8,    1)

    def forward(self, x_flat):
        x = to_cir(x_flat)                        # (N,2,500)
        x = F.relu(self.c1(x))[:, :, :500]
        x = self.d1(self.p1(x))
        x = F.relu(self.c2(x))
        x = self.flat(self.ap(x))
        x = F.relu(self.bn1(self.fc1(x)))
        x = F.relu(self.bn2(self.fc2(x)))
        x = self.d2(x)
        x = F.relu(self.bn3(self.fc3(x)))
        x = F.relu(self.fc4(x))
        return self.fc5(x).squeeze(1)


class DNNBaseline(nn.Module):
    """
    DNN from Tiemann et al. [16] – Table 2.
    Flatten(1000) → Dense(50)×3 → Dense(1)
    """
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1000, 50), nn.ReLU(),
            nn.Linear(50,   50), nn.ReLU(),
            nn.Linear(50,   50), nn.ReLU(),
            nn.Linear(50,    1)
        )

    def forward(self, x):
        return self.net(x).squeeze(1)


# Print param counts
def n_params(m): return sum(p.numel() for p in m.parameters())
print(f"\n  Model parameter counts (paper in brackets):")
print(f"    DualLossAEP  : {n_params(DualLossAEP()):>8,}   [32,019]")
print(f"    CNN baseline : {n_params(CNNBaseline()):>8,}   [271,585]")
print(f"    DNN baseline : {n_params(DNNBaseline()):>8,}   [30,800]")


# ─────────────────────────────────────────────────────────────────────────────
# 7.  TRAINING CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

# Dual-loss weights (Eq.5):  L_AEP = W_AE * L_AE  +  W_CNN * L_CNN
W_AE      = 0.3
W_CNN     = 0.7
NOISE_STD = 0.1    # σ for denoising AE Gaussian noise

LR           = 1e-3
WEIGHT_DECAY = 1e-5
MAX_EPOCHS   = 500
PATIENCE     = 25   # early-stop: no improvement for 25 epochs
MAE_FN       = nn.L1Loss()


# ─────────────────────────────────────────────────────────────────────────────
# 8.  TRAINING LOOPS
# ─────────────────────────────────────────────────────────────────────────────

def train_epoch_aep(model, loader, opt):
    model.train()
    tot_ae = tot_cnn = tot_loss = 0.0
    for Xb, yb in loader:
        Xb, yb     = Xb.to(DEVICE), yb.to(DEVICE)
        X_clean    = to_cir(Xb)
        X_noisy    = to_cir(add_noise(Xb, NOISE_STD))
        opt.zero_grad(set_to_none=True)
        recon, pred = model(X_noisy)
        l_ae  = MAE_FN(recon, X_clean)
        l_cnn = MAE_FN(pred,  yb)
        loss  = W_AE * l_ae + W_CNN * l_cnn
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        n = yb.size(0)
        tot_ae += l_ae.item()*n; tot_cnn += l_cnn.item()*n; tot_loss += loss.item()*n
    N = len(loader.dataset)
    return tot_ae/N, tot_cnn/N, tot_loss/N


@torch.no_grad()
def eval_aep(model, loader):
    model.eval()
    tot_ae = tot_cnn = tot_loss = 0.0
    for Xb, yb in loader:
        Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
        X_clean = to_cir(Xb);  X_noisy = to_cir(add_noise(Xb, NOISE_STD))
        recon, pred = model(X_noisy)
        l_ae  = MAE_FN(recon, X_clean); l_cnn = MAE_FN(pred, yb)
        loss  = W_AE * l_ae + W_CNN * l_cnn
        n = yb.size(0)
        tot_ae += l_ae.item()*n; tot_cnn += l_cnn.item()*n; tot_loss += loss.item()*n
    N = len(loader.dataset)
    return tot_ae/N, tot_cnn/N, tot_loss/N


def fit_aep(model, tr_l, val_l, tag="AEP"):
    """Full training loop for DualLossAEP. Returns best model + history."""
    model = model.to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sch   = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=10, factor=0.5, min_lr=1e-5)

    best_loss, best_state, no_imp = float("inf"), None, 0
    hist = {"tr_cnn":[], "val_cnn":[], "tr_ae":[], "val_ae":[]}

    t0 = time.time()
    for epoch in range(1, MAX_EPOCHS+1):
        tr_ae, tr_cnn, tr_tot   = train_epoch_aep(model, tr_l, opt)
        val_ae, val_cnn, val_tot = eval_aep(model, val_l)
        sch.step(val_tot)

        hist["tr_cnn"].append(tr_cnn);  hist["val_cnn"].append(val_cnn)
        hist["tr_ae"].append(tr_ae);    hist["val_ae"].append(val_ae)

        if val_tot < best_loss:
            best_loss  = val_tot
            best_state = deepcopy({k: v.cpu() for k, v in model.state_dict().items()})
            no_imp     = 0
        else:
            no_imp += 1

        if epoch % 20 == 0 or epoch == 1:
            lr_now = opt.param_groups[0]["lr"]
            elapsed = time.time() - t0
            print(f"  [{tag}] Ep {epoch:4d} | "
                  f"tr_cnn={tr_cnn:6.1f}  val_cnn={val_cnn:6.1f} mm | "
                  f"tr_ae={tr_ae:.4f}  val_ae={val_ae:.4f} | "
                  f"lr={lr_now:.1e} | patience={no_imp} | {elapsed:.0f}s")

        if no_imp >= PATIENCE:
            print(f"  [{tag}] ⏹  Early stop ep {epoch}  best_val_tot={best_loss:.4f}")
            break

    model.load_state_dict(best_state)
    return model, hist


def fit_generic(model, tr_l, val_l, tag="model"):
    """Training loop for DNN / CNN (simple MAE regression)."""
    model = model.to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sch   = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=10, factor=0.5, min_lr=1e-5)

    best_loss, best_state, no_imp = float("inf"), None, 0
    hist = {"tr":[], "val":[]}

    for epoch in range(1, MAX_EPOCHS+1):
        model.train()
        tr_loss = 0.0
        for Xb, yb in tr_l:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            loss = MAE_FN(model(Xb), yb)
            loss.backward(); opt.step()
            tr_loss += loss.item() * yb.size(0)
        tr_loss /= len(tr_l.dataset)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for Xb, yb in val_l:
                Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
                val_loss += MAE_FN(model(Xb), yb).item() * yb.size(0)
        val_loss /= len(val_l.dataset)
        sch.step(val_loss)
        hist["tr"].append(tr_loss); hist["val"].append(val_loss)

        if val_loss < best_loss:
            best_loss  = val_loss
            best_state = deepcopy({k: v.cpu() for k, v in model.state_dict().items()})
            no_imp     = 0
        else:
            no_imp += 1

        if epoch % 50 == 0 or epoch == 1:
            print(f"  [{tag}] Ep {epoch:4d} | tr={tr_loss:.1f}  val={val_loss:.1f} mm | patience={no_imp}")

        if no_imp >= PATIENCE:
            print(f"  [{tag}] ⏹  Early stop ep {epoch}  best={best_loss:.1f} mm")
            break

    model.load_state_dict(best_state)
    return model, hist


@torch.no_grad()
def get_preds(model, loader, is_aep=False):
    model.eval()
    preds, trues = [], []
    for Xb, yb in loader:
        Xb = Xb.to(DEVICE)
        if is_aep:
            _, p = model(to_cir(Xb))
        else:
            p = model(Xb)
        preds.append(p.cpu()); trues.append(yb)
    return torch.cat(preds).numpy(), torch.cat(trues).numpy()


# ─────────────────────────────────────────────────────────────────────────────
# 9.  PRE-TRAINED STACKED AE  (two-phase training, for full comparison)
# ─────────────────────────────────────────────────────────────────────────────

def fit_pretrained_ae(tr_l, val_l, tag="PreAE"):
    """
    Phase 1: train encoder+decoder unsupervised (reconstruction only).
    Phase 2: freeze encoder, train predictor supervised.
    """
    model = DualLossAEP().to(DEVICE)

    # ── Phase 1: unsupervised AE pre-training ────────────────────────────────
    print(f"\n  [{tag}] Phase 1 – unsupervised AE pre-training …")
    opt1  = torch.optim.Adam(
                list(model.encoder.parameters()) + list(model.decoder.parameters()),
                lr=LR, weight_decay=WEIGHT_DECAY)
    sch1  = torch.optim.lr_scheduler.ReduceLROnPlateau(opt1, patience=10, factor=0.5, min_lr=1e-5)
    best1_loss, best1_state, no_imp1 = float("inf"), None, 0

    for epoch in range(1, MAX_EPOCHS+1):
        model.train()
        tr_ae = 0.0
        for Xb, _ in tr_l:
            Xb = Xb.to(DEVICE)
            X_clean = to_cir(Xb);  X_noisy = to_cir(add_noise(Xb, NOISE_STD))
            opt1.zero_grad(set_to_none=True)
            recon, _ = model(X_noisy)
            loss = MAE_FN(recon, X_clean)
            loss.backward(); opt1.step()
            tr_ae += loss.item() * Xb.size(0)
        tr_ae /= len(tr_l.dataset)

        model.eval()
        val_ae = 0.0
        with torch.no_grad():
            for Xb, _ in val_l:
                Xb = Xb.to(DEVICE)
                X_clean = to_cir(Xb);  X_noisy = to_cir(add_noise(Xb, NOISE_STD))
                recon, _ = model(X_noisy)
                val_ae += MAE_FN(recon, X_clean).item() * Xb.size(0)
        val_ae /= len(val_l.dataset)
        sch1.step(val_ae)

        if val_ae < best1_loss:
            best1_loss  = val_ae
            best1_state = deepcopy({k: v.cpu() for k, v in model.state_dict().items()})
            no_imp1     = 0
        else:
            no_imp1 += 1

        if epoch % 50 == 0:
            print(f"    Phase1 Ep {epoch:4d} | tr_ae={tr_ae:.5f}  val_ae={val_ae:.5f} | patience={no_imp1}")
        if no_imp1 >= PATIENCE:
            print(f"    Phase1 ⏹  Early stop ep {epoch}")
            break

    model.load_state_dict(best1_state)

    # ── Phase 2: freeze encoder, train predictor ─────────────────────────────
    print(f"\n  [{tag}] Phase 2 – supervised predictor training (encoder frozen) …")
    for p in model.encoder.parameters():
        p.requires_grad = False

    opt2  = torch.optim.Adam(model.predictor.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sch2  = torch.optim.lr_scheduler.ReduceLROnPlateau(opt2, patience=10, factor=0.5, min_lr=1e-5)
    best2_loss, best2_state, no_imp2 = float("inf"), None, 0

    for epoch in range(1, MAX_EPOCHS+1):
        model.train()
        tr_cnn = 0.0
        for Xb, yb in tr_l:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            opt2.zero_grad(set_to_none=True)
            _, pred = model(to_cir(Xb))
            loss = MAE_FN(pred, yb)
            loss.backward(); opt2.step()
            tr_cnn += loss.item() * yb.size(0)
        tr_cnn /= len(tr_l.dataset)

        model.eval()
        val_cnn = 0.0
        with torch.no_grad():
            for Xb, yb in val_l:
                Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
                _, pred = model(to_cir(Xb))
                val_cnn += MAE_FN(pred, yb).item() * yb.size(0)
        val_cnn /= len(val_l.dataset)
        sch2.step(val_cnn)

        if val_cnn < best2_loss:
            best2_loss  = val_cnn
            best2_state = deepcopy({k: v.cpu() for k, v in model.state_dict().items()})
            no_imp2     = 0
        else:
            no_imp2 += 1

        if epoch % 50 == 0:
            print(f"    Phase2 Ep {epoch:4d} | tr_cnn={tr_cnn:.1f}  val_cnn={val_cnn:.1f} mm | patience={no_imp2}")
        if no_imp2 >= PATIENCE:
            print(f"    Phase2 ⏹  Early stop ep {epoch}")
            break

    model.load_state_dict(best2_state)
    # Un-freeze for safe re-use
    for p in model.encoder.parameters():
        p.requires_grad = True
    return model


# ─────────────────────────────────────────────────────────────────────────────
# 10. METRIC HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def report(true, pred, tag=""):
    mae_val  = float(np.abs(true - pred).mean())
    p95      = float(np.percentile(np.abs(true - pred), 95))
    raw_mae  = float(np.abs(true).mean())
    diff_pct = (raw_mae - mae_val) / raw_mae * 100
    return {"tag": tag, "mae": mae_val, "p95": p95,
            "raw_mae": raw_mae, "diff_pct": diff_pct}


def print_table(results, title):
    print(f"\n{'═'*70}")
    print(f"  {title}")
    print(f"{'═'*70}")
    print(f"  {'Approach':<22} {'MAE (mm)':>10}  {'95th pct':>10}  {'Δ vs raw':>10}")
    print(f"  {'-'*54}")
    for r in results:
        print(f"  {r['tag']:<22} {r['mae']:>10.1f}  {r['p95']:>10.1f}  {r['diff_pct']:>+9.1f}%")
    print(f"{'═'*70}")


# ─────────────────────────────────────────────────────────────────────────────
# 11. RUN MODE A – COMBINED RANDOM 80/20 SPLIT  (Table 3)
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'#'*70}")
print(f"##  MODE A – Combined Random 80/20 Split  (Table 3)  ##")
print(f"{'#'*70}")

set_seed()
tr_A, val_A, te_A, n_tr_A, n_val_A = make_loaders(Xtr_A, ytr_A, Xte_A, yte_A)
print(f"\n  Loaders: train={n_tr_A:,}  val={n_val_A:,}  test={len(yte_A):,}")

# ── (1) DNN ──────────────────────────────────────────────────────────────────
print(f"\n{'─'*50}\n  [A] DNN baseline\n{'─'*50}")
dnn_A, dnn_A_hist = fit_generic(DNNBaseline(), tr_A, val_A, tag="DNN-A")
dnn_A_pred, dnn_A_true = get_preds(dnn_A, te_A, is_aep=False)

# ── (2) CNN ──────────────────────────────────────────────────────────────────
print(f"\n{'─'*50}\n  [A] CNN standalone\n{'─'*50}")
cnn_A, cnn_A_hist = fit_generic(CNNBaseline(), tr_A, val_A, tag="CNN-A")
cnn_A_pred, cnn_A_true = get_preds(cnn_A, te_A, is_aep=False)

# ── (3) Pre-trained AE ───────────────────────────────────────────────────────
print(f"\n{'─'*50}\n  [A] Pre-trained stacked AE\n{'─'*50}")
pre_A = fit_pretrained_ae(tr_A, val_A, tag="PreAE-A")
pre_A_pred, pre_A_true = get_preds(pre_A, te_A, is_aep=True)

# ── (4) Dual-loss AEP ────────────────────────────────────────────────────────
print(f"\n{'─'*50}\n  [A] Dual-loss AEP\n{'─'*50}")
aep_A, aep_A_hist = fit_aep(DualLossAEP(), tr_A, val_A, tag="AEP-A")
aep_A_pred, aep_A_true = get_preds(aep_A, te_A, is_aep=True)

# Results
raw_A = float(np.abs(aep_A_true).mean())
res_A = [
    {"tag":"No correction",  "mae": raw_A,
     "p95": float(np.percentile(np.abs(aep_A_true), 95)),
     "raw_mae": raw_A, "diff_pct": 0.0},
    report(dnn_A_true, dnn_A_pred, "DNN [16]"),
    report(cnn_A_true, cnn_A_pred, "CNN"),
    report(pre_A_true, pre_A_pred, "Pre-trained AE"),
    report(aep_A_true, aep_A_pred, "Dual-loss AEP ★"),
]
print_table(res_A, "MODE A – Random 80/20  (paper Table 3 targets: 214.7 / 75.7 / 60.6 / 69.4 / 58.6 mm)")


# ─────────────────────────────────────────────────────────────────────────────
# 12. RUN MODE B – LEAVE-LOCATION-OUT  (Figure 6)
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'#'*70}")
print(f"##  MODE B – Leave-Location-Out  (Figure 6)  ##")
print(f"{'#'*70}")

set_seed()
tr_B, val_B, te_B, n_tr_B, n_val_B = make_loaders(Xtr_B, ytr_B, Xte_B, yte_B)
print(f"\n  Loaders: train={n_tr_B:,}  val={n_val_B:,}  test={len(yte_B):,}")

# ── (1) DNN ──────────────────────────────────────────────────────────────────
print(f"\n{'─'*50}\n  [B] DNN baseline\n{'─'*50}")
dnn_B, _ = fit_generic(DNNBaseline(), tr_B, val_B, tag="DNN-B")
dnn_B_pred, dnn_B_true = get_preds(dnn_B, te_B, is_aep=False)

# ── (2) CNN ──────────────────────────────────────────────────────────────────
print(f"\n{'─'*50}\n  [B] CNN standalone\n{'─'*50}")
cnn_B, _ = fit_generic(CNNBaseline(), tr_B, val_B, tag="CNN-B")
cnn_B_pred, cnn_B_true = get_preds(cnn_B, te_B, is_aep=False)

# ── (3) Pre-trained AE ───────────────────────────────────────────────────────
print(f"\n{'─'*50}\n  [B] Pre-trained stacked AE\n{'─'*50}")
pre_B = fit_pretrained_ae(tr_B, val_B, tag="PreAE-B")
pre_B_pred, pre_B_true = get_preds(pre_B, te_B, is_aep=True)

# ── (4) Dual-loss AEP ────────────────────────────────────────────────────────
print(f"\n{'─'*50}\n  [B] Dual-loss AEP\n{'─'*50}")
aep_B, aep_B_hist = fit_aep(DualLossAEP(), tr_B, val_B, tag="AEP-B")
aep_B_pred, aep_B_true = get_preds(aep_B, te_B, is_aep=True)

# Results
raw_B = float(np.abs(aep_B_true).mean())
res_B = [
    {"tag":"No correction",  "mae": raw_B,
     "p95": float(np.percentile(np.abs(aep_B_true), 95)),
     "raw_mae": raw_B, "diff_pct": 0.0},
    report(dnn_B_true, dnn_B_pred, "DNN [16]"),
    report(cnn_B_true, cnn_B_pred, "CNN"),
    report(pre_B_true, pre_B_pred, "Pre-trained AE"),
    report(aep_B_true, aep_B_pred, "Dual-loss AEP ★"),
]
print_table(res_B, "MODE B – Leave-Loc-Out  (paper Fig 6 CDF 95th pct: raw≈700 / AEP≈350 mm)")

# Per location breakdown (Mode B)
print(f"\n  Per-location (Mode B):")
print(f"  {'Loc':>4}  {'N':>5}  {'raw MAE':>9}  {'AEP MAE':>9}  {'Δ':>8}")
print(f"  {'-'*42}")
for loc in sorted(TEST_LOCS):
    mask = loc_B_te == loc
    raw  = float(np.abs(aep_B_true[mask]).mean())
    corr = float(np.abs(aep_B_true[mask] - aep_B_pred[mask]).mean())
    n    = mask.sum()
    print(f"  {loc:>4}  {n:>5}  {raw:>9.1f}  {corr:>9.1f}  {raw-corr:>+8.1f} mm")


# ─────────────────────────────────────────────────────────────────────────────
# 13. SAVE MODELS & NORMALISATION STATS
# ─────────────────────────────────────────────────────────────────────────────
out_dir = Path("/mnt/user-data/outputs")
out_dir.mkdir(parents=True, exist_ok=True)

torch.save({
    "model_state": {k: v.cpu() for k, v in aep_A.state_dict().items()},
    "X_mean": mu_A, "X_std": sig_A,
    "mode": "A_random_split",
    "results_A": {r["tag"]: r["mae"] for r in res_A},
    "config": {"W_AE": W_AE, "W_CNN": W_CNN, "NOISE_STD": NOISE_STD}
}, out_dir / "aep_mode_A.pt")

torch.save({
    "model_state": {k: v.cpu() for k, v in aep_B.state_dict().items()},
    "X_mean": mu_B, "X_std": sig_B,
    "mode": "B_leave_loc_out",
    "results_B": {r["tag"]: r["mae"] for r in res_B},
    "config": {"W_AE": W_AE, "W_CNN": W_CNN, "NOISE_STD": NOISE_STD}
}, out_dir / "aep_mode_B.pt")

print(f"\n✓ Saved: aep_mode_A.pt  and  aep_mode_B.pt")


# ─────────────────────────────────────────────────────────────────────────────
# 14. PUBLICATION-QUALITY PLOTS
#     Row 1 – Mode A: training curves | bar chart (Table 3) | scatter
#     Row 2 – Mode B: training curves | CDF (Figure 6)      | per-location bar
# ─────────────────────────────────────────────────────────────────────────────
COLORS = {
    "No correction":  "#888888",
    "DNN [16]":       "#4472C4",
    "CNN":            "#ED7D31",
    "Pre-trained AE": "#70AD47",
    "Dual-loss AEP ★":"#FF0000",
}
PAPER_A = {"No correction":214.7, "DNN [16]":75.7, "CNN":60.6,
           "Pre-trained AE":69.4, "Dual-loss AEP ★":58.6}

fig = plt.figure(figsize=(20, 11))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.35)
fig.suptitle("UWB Ranging Error Correction – Full Paper Replication\n"
             "Fontaine et al. (2020), IEEE Access", fontsize=13, fontweight="bold")

# ── A1: Mode A training curves ───────────────────────────────────────────────
ax = fig.add_subplot(gs[0, 0])
ax.plot(aep_A_hist["tr_cnn"],  color="#4472C4", alpha=0.5, lw=1,   label="Train CNN loss")
ax.plot(aep_A_hist["val_cnn"], color="#4472C4",            lw=1.8, label="Val CNN loss")
ax.plot(aep_A_hist["tr_ae"],   color="#ED7D31", alpha=0.5, lw=1,   label="Train AE loss")
ax.plot(aep_A_hist["val_ae"],  color="#ED7D31",            lw=1.8, label="Val AE loss")
ax.axhline(58.6, color="red", ls="--", lw=1.2, label="Paper target 58.6 mm")
ax.set_xlabel("Epoch"); ax.set_ylabel("MAE (mm / AE units)")
ax.set_title("Mode A – AEP Training Curves"); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

# ── A2: Bar chart – compare all models (Mode A, Table 3) ─────────────────────
ax = fig.add_subplot(gs[0, 1])
models_order = ["No correction", "DNN [16]", "CNN", "Pre-trained AE", "Dual-loss AEP ★"]
ours   = [r["mae"] for r in res_A]
paper  = [PAPER_A[m] for m in models_order]
x      = np.arange(len(models_order))
w      = 0.35
bars1  = ax.bar(x - w/2, ours,  w, label="This run",  color=[COLORS[m] for m in models_order], alpha=0.85)
bars2  = ax.bar(x + w/2, paper, w, label="Paper",     color=[COLORS[m] for m in models_order], alpha=0.4, hatch="//")
for bar, val in zip(bars1, ours):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2, f"{val:.0f}", ha="center", va="bottom", fontsize=7)
for bar, val in zip(bars2, paper):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2, f"{val:.0f}", ha="center", va="bottom", fontsize=7, color="gray")
ax.set_xticks(x)
ax.set_xticklabels([m.replace(" ★","") for m in models_order], rotation=20, ha="right", fontsize=8)
ax.set_ylabel("MAE (mm)"); ax.set_title("Mode A – Table 3 Comparison")
ax.legend(fontsize=8); ax.grid(True, axis="y", alpha=0.3)

# ── A3: Scatter – AEP Mode A pred vs true ────────────────────────────────────
ax = fig.add_subplot(gs[0, 2])
lim = max(np.abs(aep_A_true).max(), np.abs(aep_A_pred).max()) * 1.05
ax.scatter(aep_A_true, aep_A_pred, s=1.5, alpha=0.2, color="#4472C4")
ax.plot([-lim, lim], [-lim, lim], "r--", lw=1.5, label="Perfect")
ax.set_xlabel("True error (mm)"); ax.set_ylabel("Predicted error (mm)")
ax.set_title(f"Mode A – AEP Scatter  (MAE={res_A[-1]['mae']:.1f} mm)")
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# ── B1: Mode B training curves ───────────────────────────────────────────────
ax = fig.add_subplot(gs[1, 0])
ax.plot(aep_B_hist["tr_cnn"],  color="#4472C4", alpha=0.5, lw=1,   label="Train CNN loss")
ax.plot(aep_B_hist["val_cnn"], color="#4472C4",            lw=1.8, label="Val CNN loss")
ax.plot(aep_B_hist["tr_ae"],   color="#ED7D31", alpha=0.5, lw=1,   label="Train AE loss")
ax.plot(aep_B_hist["val_ae"],  color="#ED7D31",            lw=1.8, label="Val AE loss")
ax.set_xlabel("Epoch"); ax.set_ylabel("MAE (mm / AE units)")
ax.set_title("Mode B – AEP Training Curves"); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

# ── B2: CDF – mirrors Figure 6 exactly ───────────────────────────────────────
ax = fig.add_subplot(gs[1, 1])
cdf_curves = [
    (np.abs(aep_B_true),             "Original-ranges",       "#888888", "-",  2.0),
    (np.abs(aep_B_true-dnn_B_pred),  "DNN-corrected-ranges",  "#4472C4", "--", 1.5),
    (np.abs(aep_B_true-cnn_B_pred),  "CNN-corrected-ranges",  "#ED7D31", "-.", 1.5),
    (np.abs(aep_B_true-pre_B_pred),  "Pretrained-corrected",  "#70AD47", ":",  1.5),
    (np.abs(aep_B_true-aep_B_pred),  "AEP-corrected-ranges",  "#FF0000", "-",  2.2),
]
for errs, lbl, col, ls, lw in cdf_curves:
    se  = np.sort(errs)
    cdf = np.arange(1, len(se)+1) / len(se)
    ax.plot(se, cdf, ls, color=col, lw=lw,
            label=f"{lbl} ({errs.mean():.0f} mm)")

ax.axhline(0.95, color="black", ls=":", lw=1, label="CDF = 0.95")
ax.axvline(350,  color="red",   ls=":", lw=1, alpha=0.6, label="Paper AEP 95th=350 mm")
ax.axvline(700,  color="gray",  ls=":", lw=1, alpha=0.6, label="Paper raw  95th=700 mm")
ax.set_xlabel("Error (mm)"); ax.set_ylabel("CDF")
ax.set_title("Mode B – CDF (replication of Fig 6)")
ax.legend(fontsize=6.5, loc="lower right"); ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1100); ax.set_ylim(0, 1.02)

# ── B3: Per-location MAE bar ─────────────────────────────────────────────────
ax = fig.add_subplot(gs[1, 2])
locs_B   = sorted(TEST_LOCS)
raw_vals  = [float(np.abs(aep_B_true[loc_B_te==l]).mean()) for l in locs_B]
aep_vals  = [float(np.abs((aep_B_true-aep_B_pred)[loc_B_te==l]).mean()) for l in locs_B]
dnn_vals  = [float(np.abs((dnn_B_true-dnn_B_pred)[loc_B_te==l]).mean()) for l in locs_B]
x_locs    = np.arange(len(locs_B))
w         = 0.25
ax.bar(x_locs - w,   raw_vals,  w, label="Uncorrected",  color="#888888", alpha=0.85)
ax.bar(x_locs,       dnn_vals,  w, label="DNN corrected", color="#4472C4", alpha=0.85)
ax.bar(x_locs + w,   aep_vals,  w, label="AEP corrected", color="#FF0000", alpha=0.85)
ax.set_xticks(x_locs)
ax.set_xticklabels([f"Loc {l}" for l in locs_B])
ax.set_ylabel("MAE (mm)"); ax.set_title("Mode B – Per-Location MAE")
ax.legend(fontsize=8); ax.grid(True, axis="y", alpha=0.3)

plt.savefig(out_dir / "uwb_full_replication.png", dpi=150, bbox_inches="tight")
print(f"\n✓ Plot saved: uwb_full_replication.png")


# ─────────────────────────────────────────────────────────────────────────────
# 15. COMBINED FINAL SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'═'*70}")
print(f"  COMBINED FINAL SUMMARY")
print(f"{'═'*70}")
print(f"\n  MODE A – Random 80/20 split (Table 3)  {'Paper':>12}")
print(f"  {'-'*55}")
paper_vals_A = [214.7, 75.7, 60.6, 69.4, 58.6]
for r, pv in zip(res_A, paper_vals_A):
    diff = r['mae'] - pv
    print(f"  {r['tag']:<22} {r['mae']:>8.1f} mm     {pv:>6.1f} mm   ({diff:+.1f} mm)")

print(f"\n  MODE B – Leave-location-out (Fig 6)")
print(f"  {'-'*55}")
for r in res_B:
    print(f"  {r['tag']:<22} {r['mae']:>8.1f} mm   95th={r['p95']:>6.0f} mm")

print(f"\n  ★ Dual-loss AEP is the best model in both modes.")
print(f"  ★ Mode A should be closest to the paper's Table 3 numbers (58.6 mm target).")
print(f"  ★ Mode B 95th-pct target from Fig 6: AEP≈350 mm, Uncorrected≈700 mm.")
print(f"{'═'*70}\n")

  UWB Dual-Loss AEP – Full Paper Replication
  Device : cuda  (Tesla T4)

Loading CSVs from disk …
  Loaded 23 / 23 locations.

  Full dataset : X=(32276, 1000)  y=(32276,)
  |error| mean : 209.8 mm  (paper: 214.7 mm)

  MODE A (random 80/20): train=25,820  test=6,456
    Train |error| : 211.3 mm
    Test  |error| : 203.8 mm   ← should be ~214 mm (paper)

  MODE B (leave-loc-out): train=27,904  test=4,372
    Train locs : [1, 2, 3, 5, 6, 7, 8, 9, 10, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 23]
    Test  locs : [4, 11, 22]
    Test  |error| : 157.0 mm

  Model parameter counts (paper in brackets):
    DualLossAEP  :  648,981   [32,019]
    CNN baseline :  609,170   [271,585]
    DNN baseline :   55,201   [30,800]

######################################################################
##  MODE A – Combined Random 80/20 Split  (Table 3)  ##
######################################################################

  Loaders: train=23,238  val=2,582  test=6,456

──────────────────────────────

PermissionError: [Errno 13] Permission denied: '/mnt/user-data'